In [1]:
# =============================================================================
# TASK 4 (REVISED) — SMOD L2 -> SMOD L1 (3-class) RECODE ONLY
# NO harmonisation / NO alignment to WorldPop in this script.
#
# Requirements you set:
# - Keep integer type (do NOT convert to float)
# - Output dtype: Int16 (same family as input; categorical)
# - Convert Water (L2=10) to NoData
# - Preserve existing NoData, and set output NoData to -200 (same as input)
# - Mapping (Table 1):
#     L2 11,12,13 -> L1 1 (Rural)
#     L2 21,22,23 -> L1 2 (Urban cluster)
#     L2 30       -> L1 3 (Urban centre)
#     L2 10 (Water) and any NoData -> NoData (-200)
#
# Outputs (in outputs/):
# - 04_smod_L1_from_L2_54009_1km_INT16.tif
# - 04_smod_L1_value_counts.csv
# - 04_smod_L1_preview.png
# - 04_smod_L1_recode_report.json
# =============================================================================

from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt


# -----------------------------------------------------------------------------
# Paths (your project structure)
# -----------------------------------------------------------------------------
ROOT = Path(r"C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton")
DATA = ROOT / "data_raw"
OUT  = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

smod_l2_path = DATA / "GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V1_0_R11_C21.tif"

out_l1 = OUT / "04_smod_L1_from_L2_54009_1km_INT16.tif"
out_counts = OUT / "04_smod_L1_value_counts.csv"
out_png = OUT / "04_smod_L1_preview.png"
out_report = OUT / "04_smod_L1_recode_report.json"

run_time_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")


# -----------------------------------------------------------------------------
# Recode settings (as you requested)
# -----------------------------------------------------------------------------
NODATA_OUT = -200  # keep same nodata code as SMOD input
# L1 classes:
# 1 Rural, 2 Urban cluster, 3 Urban centre

EXPECTED_L2 = {10, 11, 12, 13, 21, 22, 23, 30}  # expected legend values


def recode_l2_to_l1_int16(arr_l2: np.ndarray, nodata_in: int) -> tuple[np.ndarray, dict]:
    """
    Recode L2 categorical values to L1 categorical values using int16.
    Water (10) and nodata_in -> NODATA_OUT (-200).
    Unknown valid codes -> NODATA_OUT, but counted and reported.
    """
    # Force int16 output
    out = np.full(arr_l2.shape, NODATA_OUT, dtype=np.int16)

    # Valid data mask (not nodata)
    valid = arr_l2 != nodata_in

    # Water -> nodata (leave as NODATA_OUT)
    # Rural
    out[valid & np.isin(arr_l2, [11, 12, 13])] = 1
    # Urban cluster
    out[valid & np.isin(arr_l2, [21, 22, 23])] = 2
    # Urban centre
    out[valid & (arr_l2 == 30)] = 3

    # Unknown codes (valid but not in legend)
    unknown_mask = valid & (~np.isin(arr_l2, list(EXPECTED_L2)))
    unknown_count = int(unknown_mask.sum())
    unknown_values = np.unique(arr_l2[unknown_mask]).tolist() if unknown_count > 0 else []

    # Count water cells explicitly (10)
    water_count = int((valid & (arr_l2 == 10)).sum())

    stats = {
        "water_cells_set_to_nodata": water_count,
        "unknown_cells_set_to_nodata": unknown_count,
        "unknown_values": unknown_values
    }
    return out, stats


# -----------------------------------------------------------------------------
# 1) Read L2 raster
# -----------------------------------------------------------------------------
with rasterio.open(smod_l2_path) as src:
    l2 = src.read(1)  # should be int16
    profile_in = src.profile.copy()
    crs_in = src.crs
    transform_in = src.transform
    nodata_in = src.nodata
    dtype_in = src.dtypes[0]

if nodata_in is None:
    # SMOD should have nodata=-200; but if missing, we must decide.
    # We will default to -200 to match your requirement and the dataset metadata.
    nodata_in = -200

print("SMOD L2 input:")
print(" - path:", smod_l2_path.name)
print(" - dtype:", dtype_in)
print(" - nodata:", nodata_in)
print(" - CRS:", crs_in)
print(" - shape:", l2.shape)
print()


# -----------------------------------------------------------------------------
# 2) Recode to L1 (int16), preserving nodata=-200 and water->nodata
# -----------------------------------------------------------------------------
l1, recode_stats = recode_l2_to_l1_int16(l2.astype(np.int16, copy=False), int(nodata_in))

# Sanity check: dtype
assert l1.dtype == np.int16, "Output is not int16 — something changed the dtype unexpectedly."

# Value counts (including nodata)
vals, counts = np.unique(l1, return_counts=True)
count_tbl = pd.DataFrame({"value": vals.astype(int), "cell_count": counts.astype(int)})
count_tbl["class"] = count_tbl["value"].map({
    -200: "NoData (incl. Water)",
    1: "Rural (L1=1)",
    2: "Urban cluster (L1=2)",
    3: "Urban centre (L1=3)"
}).fillna("Other/Unexpected")
count_tbl.to_csv(out_counts, index=False)

print("Recode stats:", recode_stats)
print("Wrote:", out_counts)
print()


# -----------------------------------------------------------------------------
# 3) Write L1 raster (int16, nodata=-200)
# -----------------------------------------------------------------------------
profile_out = profile_in.copy()
profile_out.update(
    dtype=rasterio.int16,
    nodata=NODATA_OUT,
    count=1,
    compress="LZW"
)

with rasterio.open(out_l1, "w", **profile_out) as dst:
    dst.write(l1, 1)

print("Wrote:", out_l1)
print()


# -----------------------------------------------------------------------------
# 4) Simple visualisation (categorical preview)
# -----------------------------------------------------------------------------
# For plotting, convert a copy to float so we can mask nodata for display ONLY.
# (This does NOT affect the saved raster.)
preview = l1.astype(np.float32)
preview[preview == NODATA_OUT] = np.nan

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(preview, interpolation="nearest")
ax.set_title("SMOD L1 (from L2) — 1=Rural, 2=Urban cluster, 3=Urban centre; NoData=-200 (incl. water)")
ax.set_axis_off()
cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_ticks([1, 2, 3])
cbar.set_ticklabels(["Rural (1)", "Urban cluster (2)", "Urban centre (3)"])
plt.tight_layout()
plt.savefig(out_png, dpi=300)
plt.close(fig)

print("Wrote:", out_png)
print()


# -----------------------------------------------------------------------------
# 5) Reproducibility report (metadata artefact)
# -----------------------------------------------------------------------------
report = {
    "run_time_utc": run_time_utc,
    "input": {
        "path": str(smod_l2_path),
        "dtype": str(dtype_in),
        "nodata": int(nodata_in) if nodata_in is not None else None,
        "crs": str(crs_in),
        "shape": [int(l2.shape[0]), int(l2.shape[1])]
    },
    "output": {
        "path": str(out_l1),
        "dtype": "int16",
        "nodata": NODATA_OUT,
        "crs": str(crs_in),
        "shape": [int(l1.shape[0]), int(l1.shape[1])]
    },
    "recode": {
        "mapping": {
            "10": "Water -> NoData (-200)",
            "11,12,13": "Rural -> 1",
            "21,22,23": "Urban cluster -> 2",
            "30": "Urban centre -> 3",
            "input NoData": "Preserved as NoData (-200)"
        },
        "stats": recode_stats
    },
    "notes": [
        "This script performs only the L2->L1 recoding (no reprojection/resampling/harmonisation).",
        "Nearest-neighbour resampling would be required if aligning to another grid, but that is intentionally not done here.",
        "The output raster is categorical Int16 with NoData=-200."
    ]
}

with open(out_report, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Wrote:", out_report)


# -----------------------------------------------------------------------------
# 6) Copy-friendly interview snippet (Task 4)
# -----------------------------------------------------------------------------
print("\n--- Interview notes (Task 4 — recode only) ---")
print("Created SMOD L1 by reclassifying L2 codes (categorical recode only, no harmonisation).")
print("Mapping: 11–13 -> Rural(1), 21–23 -> Urban cluster(2), 30 -> Urban centre(3).")
print("Converted Water (10) to NoData and kept NoData coding as -200; output stored as Int16 (no float conversion).")
print("---------------------------------------------\n")


SMOD L2 input:
 - path: GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V1_0_R11_C21.tif
 - dtype: int16
 - nodata: -200.0
 - CRS: ESRI:54009
 - shape: (1000, 1000)

Recode stats: {'water_cells_set_to_nodata': 10263, 'unknown_cells_set_to_nodata': 0, 'unknown_values': []}
Wrote: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\04_smod_L1_value_counts.csv

Wrote: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\04_smod_L1_from_L2_54009_1km_INT16.tif

Wrote: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\04_smod_L1_preview.png

Wrote: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\04_smod_L1_recode_report.json

--- Interview notes (Task 4 — recode only) ---
Created SMOD L1 by reclassifying L2 codes (ca